<a href="https://colab.research.google.com/github/eduardozurek/ec1/blob/main/_001_contador_binario_VHDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manejo del Reloj de 50 MHz en la board Altera DE2-115
Este ejemplo hace parte del curso de Estructura del Computador I del programa de Ingeniería de Sistemas de la Universidad del Norte.


Eduardo Zurek, Ph.D.

Estructura del Computador I

Ingeniería de Sistemas

Universidad del Norte

Este ejemplo muestra la implementación de un contador binario de 3 bits, la configuración inicial del reloj del contador se hará para que el conteo ocurra con una frecuencia de 2 Hz. A continuación se muestra el script de VHDL que implementa el contador, en las celdas posteriores explicará detalladamente la solución.

```vhdl
library IEEE;
use IEEE.STD_LOGIC_1164.ALL;
use IEEE.NUMERIC_STD.ALL;

entity contador_3bits is
    Port (
        CLOCK_50 : in  STD_LOGIC;
        RESET    : in  STD_LOGIC;
        Q        : out STD_LOGIC_VECTOR(2 downto 0)
    );
end contador_3bits;

architecture Behavioral of contador_3bits is

    -- Divisor para obtener una actualización cada 0.5 segundos
    signal divisor : integer range 0 to 24_999_999 := 0;

    -- Contador de 3 bits
    signal contador : unsigned(2 downto 0) := "000";

begin

    process(CLOCK_50)
    begin
        if rising_edge(CLOCK_50) then

            if RESET = '1' then
                divisor  <= 0;
                contador <= "000";

            else

                if divisor = 24_999_999 then
                    divisor <= 0;
                    contador <= contador + 1;
                else
                    divisor <= divisor + 1;
                end if;

            end if;

        end if;
    end process;

    Q <= std_logic_vector(contador);

end Behavioral;
```

## Las Bibliotecas (library)

```vhdl
library IEEE;
use IEEE.STD_LOGIC_1164.ALL;
use IEEE.NUMERIC_STD.ALL;
```
Estas tres líneas permiten utilizar en VHDL los tipos de datos y operaciones estandarizadas por IEEE. Conviene distinguir entre library y use, porque no hacen exactamente lo mismo.

```vhdl
library IEEE;
```
Indica al compilador que el diseño utilizará la biblioteca IEEE.

Una biblioteca en VHDL es una colección de paquetes previamente compilados. IEEE contiene varios de los paquetes estándar más utilizados en diseño digital.

Esta línea no importa todavía ningún tipo u operación concreta; simplemente hace visible la biblioteca para poder referenciar sus paquetes.


```
use IEEE.STD_LOGIC_1164.ALL;
```

Importa el contenido del paquete STD_LOGIC_1164.

Este paquete define los tipos fundamentales:

```
STD_LOGIC
STD_LOGIC_VECTOR
```
Por ejemplo, en nuestro contador:

```
CLOCK_50 : in STD_LOGIC;
```

representa una señal digital individual.

Mientras que:

```
LEDR : out STD_LOGIC_VECTOR(2 downto 0);
```

representa un conjunto de 3 señales digitales:
```text
LEDR(2)  LEDR(1)  LEDR(0)
   │        │        │
   ▼        ▼        ▼
   0/1      0/1      0/1
```

Una característica importante es que STD_LOGIC no solamente puede representar 0 y 1. El estándar define 9 valores:


| Valor | Significado |
|:-----:|-------------|
| `'0'` | cero lógico |
| `'1'` | uno lógico |
| `'U'` | no inicializado |
| `'X'` | desconocido |
| `'Z'` | alta impedancia |
| `'W'` | desconocido débil |
| `'L'` | cero débil |
| `'H'` | uno débil |
| `'-'` | don't care |

En FPGA normalmente trabajamos principalmente con '0' y '1', pero los demás estados son especialmente útiles durante la simulación.

```
use IEEE.NUMERIC_STD.ALL;
use IEEE.NUMERIC_STD.ALL;
```

Este paquete proporciona tipos y operaciones para realizar aritmética binaria.

Los dos tipos más importantes son:
```
UNSIGNED
SIGNED
```
En nuestro diseño utilizamos:
```
signal contador : unsigned(2 downto 0) := "000";
```
Esto le indica a VHDL que esos tres bits representan un número binario sin signo:

```
bits          decimal

000              0
001              1
010              2
011              3
100              4
101              5
110              6
111              7
```

Gracias a NUMERIC_STD podemos escribir directamente:
```
contador <= contador + 1;
```
El operador + está definido para UNSIGNED.

## El Proyecto

Para continuar con este ejercicio, cree un folder vacío llamado contador_3bits, dentro de ese folder cree un proyecto de Quartus llamado contador_3bits.

En la creación del proyecto defina:

```
Device Family: Cyclone IV E
Device Name: EP4CE115F29C7
```
Despúes de crear el proyecto, cree un archivo VHDL que contenga las instrucciones indicadas al principio de este cuaderno.

Y luego:

Processing ➜ Start ➜ Start Analysis & Synthesis



### El Pin Planner

Al finalizar exitosamente el proceso de análisis y síntesis, podemos configurar el Pin Planner:

Assignments ➜ Pin Planner

El reloj principal de 50 MHz es:
```
Señal VHDL	Pin FPGA	Función	I/O Standard
CLOCK_50	PIN_Y2	Reloj de 50 MHz	3.3-V LVTTL
```
La DE2-115 dispone de 9 LED verdes, LEDG[0] a LEDG[8]:

```
Señal	Pin FPGA
LEDG[0]	PIN_E21
LEDG[1]	PIN_E22
LEDG[2]	PIN_E25
LEDG[3]	PIN_E24
LEDG[4]	PIN_H21
LEDG[5]	PIN_G20
LEDG[6]	PIN_G22
LEDG[7]	PIN_G21
LEDG[8]	PIN_F17
```
Para esos LED, el manual indica 2.5 V como estándar de I/O.

La tarjeta tiene 18 switches, SW[0] a SW[17]:
```
Señal	Pin FPGA
SW[0]	PIN_AB28
SW[1]	PIN_AC28
SW[2]	PIN_AC27
SW[3]	PIN_AD27
SW[4]	PIN_AB27
SW[5]	PIN_AC26
SW[6]	PIN_AD26
SW[7]	PIN_AB26
SW[8]	PIN_AC25
SW[9]	PIN_AB25
SW[10]	PIN_AC24
SW[11]	PIN_AB24
SW[12]	PIN_AB23
SW[13]	PIN_AA24
SW[14]	PIN_AA23
SW[15]	PIN_AA22
SW[16]	PIN_Y24
SW[17]	PIN_Y23
```

Para este ejemplo, usaremos Y2 como el CLK de 50 MHz, LEDG del 0 al 2 como salida del contador, y el SW[17] como señal de reset.

### Compilación definitiva y ejecución en la board

Una vez configurado el Pin Planner, procedemos con la compilación:

Processing ➜ Start Compilation

Luego, abra el Programmer:

Tools ➜ Programmer

Verifique que la board esté conectada y aplique Start.

Al ejecutar la implementación debe ver en la board el conteo aparecer el los tres LEDs verdes más a la derecha, para resetear el conteo, opere el switche que esté más a la izquierda.
